In [ ]:
import scanpy as sc
import anndata as ad
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import harmonypy as hm
import warnings
warnings.filterwarnings('ignore')

## 1) Load data

In [ ]:
import re

# standardize probe name
def canonicalize_names(names):
    out = []
    for n in names:
        n = n.replace(':', '-').replace('.', '-').replace('_', '-').replace('+', '-plus-')
        out.append(n)
    return out

def is_variant_probe(name):
    """Variant probes contain '-WT' or '-ALT' (after canonicalization)."""
    return ('-WT' in name) or ('-ALT' in name)

# an example of loading data
aml_path = '/diskmnt/primary/Xenium/data/20251015__164339__20251015_5K_PDAC_CH/output-XETG00523__0074741__CH002Ta1-Tk1Fp1U2__20251015__164752'
nbm_path = '/diskmnt/primary/Xenium/data/20260129__192535__20260129_5K_MDS_Normal_Bone/output-XETG00611__0093316__SN078H1-Ma1Fd2U3__20260129__193040'

def load_xenium(path, sample_id):
    adata = sc.read_10x_h5(f'{path}/cell_feature_matrix.h5')
    cells_df = pd.read_parquet(f'{path}/cells.parquet')
    adata.obs = cells_df.set_index(adata.obs_names)
    adata.obsm['spatial'] = cells_df[['x_centroid', 'y_centroid']].values
    adata.obs['sample_id'] = sample_id
    # standardize feature names BEFORE intersecting
    adata.var_names = pd.Index(canonicalize_names(adata.var_names.tolist()))
    adata.var_names_make_unique()
    # Prefix cell barcodes by sample_id so obs_names are unique across samples
    adata.obs_names = pd.Index([f"{sample_id}_{n}" for n in adata.obs_names])
    return adata

aml = load_xenium(aml_path, 'AML')
nbm = load_xenium(nbm_path, 'NBM')
print(f"AML: {aml.shape[0]} cells, {aml.shape[1]} features ({sum(is_variant_probe(n) for n in aml.var_names)} variant probes)")
print(f"NBM: {nbm.shape[0]} cells, {nbm.shape[1]} features ({sum(is_variant_probe(n) for n in nbm.var_names)} variant probes)")
logging.info(f'Loaded AML {aml.shape}, NBM {nbm.shape}')

AML: 88320 cells, 5101 features (88 variant probes)
NBM: 63942 cells, 5101 features (5 variant probes)


## 2) Pre-processing & normalization using Pearson residuals

In [ ]:
# Save raw counts (needed for scANVI and per-sample Scanorama normalization)
adata.layers['counts'] = adata.X.copy()

# Pearson residuals: variance-stabilizing normalization (analytic SCTransform)
sc.experimental.pp.normalize_pearson_residuals(adata, theta=100)

# HVG by residual variance (top 2000)
X = adata.X if isinstance(adata.X, np.ndarray) else adata.X.toarray()
gene_var = np.var(X, axis=0)
top_idx = np.argsort(gene_var)[::-1][:2000]
hvg_mask = np.zeros(adata.shape[1], dtype=bool)
hvg_mask[top_idx] = True
adata.var['highly_variable'] = hvg_mask
print(f'HVGs: {hvg_mask.sum()} / {adata.shape[1]}')

# PCA (no scaling — Pearson residuals already variance-stabilized)
sc.tl.pca(adata, n_comps=30, mask_var='highly_variable')
logging.info('PCA done (Pearson residuals, 2000 HVG, 30 PCs)')

HVGs: 2000 / 5002


## 3) scANVI integration

In [ ]:
import scvi

adata_scvi = adata.copy()
adata_scvi.X = adata_scvi.layers['counts'].copy()

scvi.model.SCVI.setup_anndata(adata_scvi, batch_key='sample_id')
vae = scvi.model.SCVI(adata_scvi, n_latent=30, n_layers=2)
vae.train(max_epochs=10, batch_size=512)                            # if you have GPU which should be 10-15x faster, you need to specify it: vae.train(max_epochs=10, batch_size=512, accelerator='gpu')

adata_scvi.obs['leiden_init'] = adata.obs['leiden'].values
scanvi_model = scvi.model.SCANVI.from_scvi_model(vae, labels_key='leiden_init', unlabeled_category='Unknown')
scanvi_model.train(max_epochs=10, batch_size=512)

adata.obsm['X_scANVI'] = scanvi_model.get_latent_representation()

sc.pp.neighbors(adata, use_rep='X_scANVI', n_neighbors=15, metric='euclidean')
sc.tl.umap(adata, min_dist=0.05)
sc.tl.leiden(adata, resolution=1.0, key_added='leiden_scanvi')

adata.obsm['X_umap_scanvi'] = adata.obsm['X_umap'].copy()
logging.info('scANVI done')

In [ ]:
# UMAP
sc.pl.umap(adata, color=[ "leiden_scanvi"], legend_loc="on data", frameon=False, ncols=2)

In [ ]:
# save
adata.write_h5ad("path_to_your_output_folder/adata.h5ad")